# Chapter 9 explore: Hybrid System — Combining Fine-Tuning with Retrieval

Interactive companion to `code/chapter_09/hybrid_rag_finetune.py`. Builds a BM25 retrieval index over the full archive (held-out report included -- retrieval isn't training) and combines it with Chapter 8's fine-tuned checkpoint. Needs Chapter 8's checkpoint to exist first (`python code/chapter_08/finetune_at_scale.py`, ~30 min).

In [ ]:
import sys
sys.path.insert(0, "../code/chapter_01")
sys.path.insert(0, "../code/chapter_02")
sys.path.insert(0, "../code/chapter_06")
sys.path.insert(0, "../code/chapter_07")
sys.path.insert(0, "../code/chapter_09")

from load_local_model import MODEL_NAME, load_model_and_tokenizer, generate_reply
from hybrid_rag_finetune import build_retrieval_corpus, build_bm25_index, retrieve, answer_with_retrieval, latest_checkpoint
from peft import PeftModel

corpus = build_retrieval_corpus()
bm25 = build_bm25_index(corpus)
print(f"{len(corpus)} retrievable chunks across {len({c['report_num'] for c in corpus})} reports")

Try your own query -- see which report and time window it retrieves.

In [ ]:
query = "circulate to cool the well after running logs"  # try your own
for r in retrieve(query, corpus, bm25, k=5):
    print(f"  [{r['score']:.2f}] Report #{r['report_num']} {r['from_time']}-{r['to_time']}: {r['text'][:80]}")

Load Chapter 8's fine-tuned checkpoint and compare an answer with and without retrieval.

In [ ]:
model, tokenizer = load_model_and_tokenizer(MODEL_NAME)
lora_model = PeftModel.from_pretrained(model, latest_checkpoint())

instruction = "What happened on this well during this time window?"
input_context = "Well: FORGE 16A [78]-32 | Report #37 | Time: 20:30-21:30"

print("Without retrieval:")
print(" ", generate_reply(lora_model, tokenizer, f"{instruction}\n{input_context}", max_new_tokens=60))

print("\nWith retrieval:")
result = answer_with_retrieval(lora_model, tokenizer, instruction, input_context, query, corpus, bm25)
print(" ", result["answer"])
print("  Sources:", result["sources"])